##### Ingress

In [ ]:
helm repo add ingress-nginx https://kubernetes.github.io/ingress-nginx
helm repo update

In [ ]:
vim nginx-ingress-values.yaml

In [ ]:
controller:
  replicaCount: 2
  hostNetwork: true
  dnsPolicy: ClusterFirstWithHostNet

  hostPort:
    enabled: true
    ports:
      http: 80
      https: 443

  service:
    type: LoadBalancer
    loadBalancerIP: 172.16.6.90

  admissionWebhooks:
    enabled: false

  metrics:
    enabled: true

  podDisruptionBudget:
    enabled: true


upgrade

In [ ]:
cat > nginx-ingress-values.yaml <<EOF
controller:
  kind: DaemonSet          # ← change from Deployment to DaemonSet
  hostNetwork: true
  dnsPolicy: ClusterFirstWithHostNet
  hostPort:
    enabled: true
    ports:
      http: 80
      https: 443
  service:
    type: LoadBalancer
    loadBalancerIP: 172.16.6.90
  admissionWebhooks:
    enabled: false
  metrics:
    enabled: true
  podDisruptionBudget:
    enabled: true
  tolerations:             # ← add this to run on master nodes
  - key: "node-role.kubernetes.io/control-plane"
    operator: "Exists"
    effect: "NoSchedule"
EOF

In [ ]:
helm upgrade --install ingress-nginx ingress-nginx/ingress-nginx \
  --namespace ingress-nginx \
  --create-namespace \
  -f nginx-ingress-values.yaml

In [ ]:
kubectl get svc -n ingress-nginx -w
kubectl rollout status deployment ingress-nginx-controller -n ingress-nginx